In [2]:
# ========== 导入：代码讲解助手需要的工具 ==========

# 标准库 os：读环境变量（本笔记本主要用本地 Ollama，os 预留扩展）
import os
# dotenv：若后续要读 .env 密钥可用（本格先导入）
from dotenv import load_dotenv
# IPython 展示：Markdown 渲染、display / update_display 做流式刷新
from IPython.display import Markdown, display, update_display
# OpenAI SDK：这里指向本地 Ollama 的 OpenAI 兼容接口（/v1）
from openai import OpenAI
# requests：用 HTTP 探测 Ollama 是否在跑
import requests


In [3]:
# ========== 健康检查：本机 Ollama 是否监听 11434 ==========

# GET 根路径；能拿到 content 通常说明 Ollama 进程在跑（URL 勿改）
requests.get("http://localhost:11434").content


b'Ollama is running'

In [4]:
# ========== Ollama 的 OpenAI 兼容 API 基址 ==========

# /v1 表示走 OpenAI 风格路径（chat.completions），不是 Ollama 原生 /api/chat
OLLAMA_BASE_URL = "http://localhost:11434/v1"


In [5]:
# ========== 客户端常量：本地 Llama ==========

# base_url 指向本机 Ollama；api_key 对本地服务常可填占位字符串 'ollama'
ollama = OpenAI(base_url=OLLAMA_BASE_URL, api_key='ollama')
# 模型名须与本机 `ollama list` 里已拉取的名字一致（字符串勿改）
MODEL_LLAMA = 'llama3.2'


In [6]:
# ========== 系统提示词：技术讲解助手的回答规范 ==========
# 变量名保持原样 system_promt（原作者拼写）；prompt 英文内容勿翻译

system_promt= """ 
                You are a technical explainer and problem-solver.
Your job is to answer the user’s technical question with a clear, accurate explanation that matches their skill level.

Guidelines:
- Start with a short direct answer.
- Then explain the concept step-by-step in plain language.
- Use a small example when helpful (code, command, or pseudo-code).
- Call out assumptions and edge cases.
- If the question is ambiguous, ask 1–2 clarifying questions, otherwise make reasonable assumptions and state them.
- Be honest about uncertainty; do not invent facts.
- Prefer practical guidance over theory unless the user asks for theory.
Output format:
1) Direct answer
2) Explanation
3) Example (if useful)
4) Common pitfalls / gotchas
5) Next steps (optional)

Respond in markdown with proper label and sub label without code blocks.

"""


In [7]:
# ========== 构造 user prompt：请模型「解释这段代码」 ==========

# 传入一段代码/问题字符串，拼成固定英文前缀 + 内容（前缀影响回答，勿改）
def get_question_user_prompt(question):
    # 三引号模板：固定英文指令 + 换行，后面再拼接具体代码
    question_user_prompt = """ Please explain what this code does and why:
                                   \n """
    # 把调用方传入的 question（通常是代码片段）追加到提示词末尾
    question_user_prompt +=question 
    # 返回完整 user 消息内容，供 chat.completions 使用
    return question_user_prompt
  
#           


In [8]:
# ========== 示例问题：集合推导式 + yield from ==========

# 待讲解的代码片段（影响模型输入的字符串，保持原样）
question=' yield from {book.get("author") for book in books if book.get("author")} '
# 调用上面的函数，预览拼好的 user prompt（笔记本里会显示返回值）
get_question_user_prompt(question)


' Please explain what this code does and why:\n                                   \n  yield from {book.get("author") for book in books if book.get("author")} '

In [9]:
# ========== 流式调用 Ollama，并在单元格里实时刷新 Markdown ==========

def stream_code_explainer_ollama(question):
    # 向本地 Llama 发起流式 Chat Completions
    stream = ollama.chat.completions.create(
        # 使用上面定义的本地模型名
        model=MODEL_LLAMA,
        messages=[
            # system：讲解规范（注意原变量名 system_promt）
            {"role": "system", "content": system_promt},
            # user：固定「请解释代码」前缀 + 具体 question
            {"role": "user", "content": get_question_user_prompt(question)}
          ],
        # stream=True：按块（chunk）推送，边收边显示
        stream=True  # streanms one by one in chunks (parts)
    )    
    # 累积已生成文本
    response = ""
    # 先放一个空 Markdown 占位，拿到 display_id 以便后续原地更新
    display_handle = display(Markdown(""), display_id=True)
    # 逐块拼接，并用 update_display 刷新同一输出区（打字机效果）
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        update_display(Markdown(response), display_id=display_handle.display_id)


In [10]:
# ========== 跑一遍：用 Llama 3.2 讲解上面的示例 question ==========

# 复用上一格的 question；调用流式讲解函数，结果会显示在本格输出区
stream_code_explainer_ollama(question)


**Direct Answer**
The given code uses a generator expression to extract the authors of books from a list of dictionaries.


**Explanation**

This code is a mix of two advanced concepts: iterators and dictionary methods. Here's how it works:

*   The `yield from` expression is used to delegate sub-iteration for the resulting iterator.
*   The `.get("author") for book in books if book.get("author")` part is a filter. It filters out any dictionaries (`book`) that don't contain an "author" key.

**Example**

Suppose we have a list of dictionaries representing books, where each dictionary has an "author" key:

```python
books = [
    {"title": "Book One", "author": "Author1"},
    {"title": "Book Two", "author": "Author2"},
    {"title": "Another Book"}
]
```

The code would process the first two books, because they contain the "author" key:

```python
authors = yield from {book.get("author") for book in books if book.get("author")}
# authors -> ['Author1', 'Author2']
```

But it would skip the third, since its dictionary didn't contain an "author" key.


**Common Pitfalls / Gotchas**

*   If you are not careful, using `.get()` can result in keys being treated as attributes with non-dictionary values. In other words, if a book object is missing the author key and has `book.author`, things won't behave correctly.
*   Don’t forget to handle exceptions while using `.get()`. There will always be potential edge cases.

**Next Steps**
Consider replacing `'Book'` in code with an array or list containing strings to further process these filtered book names.

In [11]:
# ========== 再试：来自 week1 练习的代码片段 ==========

# 换一段新代码作为 question（字符串内容保持原样，含引号嵌套）
question=""""week1 EXERCISE.ipynb"links = fetch_website_links(url)
           user_prompt += "\n".join(links) """
# 同样走流式讲解路径
stream_code_explainer_ollama(question)


**Direct Answer**
This code downloads a list of links from a webpage, appends the links to a string, and saves the response for later use.

**Explanation**

This code snippet is part of a larger script that interacts with a website. Here's a step-by-step breakdown:

1. `fetch_website_links(url)`: This function fetches (downloads) a list of links from a given URL on the internet.
2. `week1 EXERCISE.ipynb links = ...`: The `links` variable contains the returned result of `fetch_website_links(url).
3. `" ".join(links)`: This expression concatenates all the links in the `links` list into a single string, separated by spaces.
4. `user_prompt += ...`: The resulting string is appended to a larger string called `user_prompt`.

**Example**

```python
def fetch_website_links(url):
    # Replace this with your actual website URL and scraping logic
    return ["https://www.example.com", "https://www.google.com"]

links = fetch_website_links("https://www.example.com")
new_links_string = " ".join(links)
user_prompt += new_links_string
```

**Common Pitfalls / Gotchas**

* Make sure you're respecting the website's `robots.txt` file and scraping policies to avoid getting banned or reported.
* Be cautious when dealing with dynamically generated content, as it might require additional logging, JavaScript execution, or user interaction.

Next Steps:

Since you've asked for an explanation, if your question has more context (like what "fetch_website_links" function does or what kind of data is stored in `user_prompt`), I'd love to ask a few clarifying questions:

* Can you explain where this script runs (e.g., Jupyter Notebook, command-line interface)?
* What is the purpose of `user_prompt` in this context?
* Does the code handle errors or edge cases (e.g., non-HTTP responses or server-side scraping restrictions)?

### 使用 Gradio 添加 UI 界面

上面几格是「在笔记本单元格里流式显示」；从下一格开始，把同一套讲解逻辑接到 **Gradio 聊天窗口**，方便反复提问。

启发式函数 `looks_like_code` 会判断用户输入是否像代码：像代码就用讲解 system prompt，否则退回普通助手。


In [22]:
# ========== 额外依赖：Gradio UI + 正则启发式 ==========

# Gradio：搭 ChatInterface 网页聊天
import gradio as gr
# re：用正则模式给「像不像代码」打分
import re


In [25]:
# ========== 启发式：输入是否像代码（命中模式数 ≥ 2 则判定为代码） ==========

def looks_like_code(s: str) -> bool:
    # 去掉首尾空白，避免空格干扰判断
    s = s.strip()
    # 空字符串直接不是代码
    if not s:
        return False

    # 若干「代码味」特征；每条命中计 1 分（行尾英文注释保留，属可运行旁注）
    patterns = [
        r"```",                          # fenced code blocks
        r"^\s{4,}\S",                    # indented lines (4+ spaces)
        r"\b(def|class|import|from|if|elif|else|for|while|try|except|with|return|print)\b",
        r"[(){}\[\];]",                  # code punctuation
        r"==|!=|<=|>=|:=|\+=|-=|\*=|/=|//=|\*\*",
        r":\s*$",                        # line ending with colon (Python blocks)
        r"#",                            # comment
    ]

    # MULTILINE：让 ^/$ 按行匹配；对每个 pattern 做 search，True 记 1
    score = sum(bool(re.search(p, s, re.MULTILINE)) for p in patterns)
    # 阈值 2：降低误报（单看一个 # 或一对括号不够）
    if score >= 2:
        likely_code=True
    else:
        likely_code=False

    # 返回布尔结果，供聊天回调分支使用
    return likely_code





In [ ]:
# ========== Gradio 回调：代码走讲解提示词，否则普通助手 ==========

def generate_code_explainer_ollama(message,history):
    # 把 Gradio 历史压成 role/content 列表
    history=[{"role":h['role'],"content":h['content']}for h in history]
    # 像代码 → 用技术讲解 system + 「解释代码」user 模板
    if looks_like_code(message)==True:
        relative_system_promt=system_promt
        relative_user_prompt=get_question_user_prompt(message)
    else:
        # 不像代码 → 普通助手 system；user 直接用原文（英文 system 字符串勿改）
        relative_system_promt='You are an polite helpful assistant'
        relative_user_prompt=message
        
    
    # 流式调用本地 Llama：system + 历史 + 当前 user
    stream = ollama.chat.completions.create(
        model=MODEL_LLAMA,
        messages=[{"role": "system", "content": relative_system_promt}] +history+
          [{"role": "user", "content": relative_user_prompt}],
        stream=True  # streanms one by one in chunks (parts)
    )    
    # 累积并逐步 yield，驱动 Gradio 打字机更新
    response = ""
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        yield response


In [27]:
# ========== 启动 Gradio「代码讲解」聊天界面 ==========

# title 保持英文原样（界面标题字符串，勿改）
chat=gr.ChatInterface(fn=generate_code_explainer_ollama,title='Code Explainer')
# share=True：尝试生成临时公网链接（需 Gradio 分享服务可用）
chat.launch(share=True)


* Running on local URL:  http://127.0.0.1:7871
* Running on public URL: https://f57e517315f0b2f1b5.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
